# Aula 2 — Análise Exploratória, Parte 1

> **Data:** 03/09  
> **Professor(a):** Felipe Aldrighi e Sofia Sayuri


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Limpeza e tratamento dos dados

Para esta aula, utilizaremos a base de dados do **Titanic**! Essa base tem dados sobre os passageiros do famoso transatlântico, incluindo a informação de que sobreviveram ou não, e é um case de estudo famoso na comunidade de Ciência de Dados.

**Dicionário - Titanic DataSet**

- `PassengerId`: Código de identificação do passageiro;
- `Survived`: Se sobreviveu (survived=1) ou não (survived=0);
- `Pclass`: Classe comercial. Pode ser primeira (Pclass=1), segunda (Pclass=2) ou terceira classe (Pclass=3);
- `Name`: Nome do passageiro;
- `Sex`: Sexo biológico;
- `Age`: Idade;
- `SibSp`: Número de irmãos;
- `Parch`: Representa o número de pais ou crianças que o passageiro tinha consigo a bordo;
- `Ticket`: Número do ticket;
- `Fare`: Valor pago pelo ticker;
- `Cabin`: Cabine onde ficou hospedado;
- `Embarked`: Local onde embarcou (C = Cherbourg; Q = Queenstown; S = Southampton).

In [ ]:
import shutil
import urllib.request
from pathlib import Path

# Na primeira vez a base é baixada para data_raw/ (o original, que fica de
# referência) e copiada para data/ (a cópia de trabalho, que é a que vamos usar).
# Bagunçou a base? Apague data/titanic.csv e rode esta célula de novo.
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
Path('data_raw').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)

if not Path('data_raw/titanic.csv').exists():
    print('Baixando titanic.csv...')
    urllib.request.urlretrieve(url, 'data_raw/titanic.csv')
if not Path('data/titanic.csv').exists():
    shutil.copy('data_raw/titanic.csv', 'data/titanic.csv')

df = pd.read_csv('data/titanic.csv')

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
95,987,0,3,"Tenglin, Mr. Gunnar Isidor",male,25.0,0,0,350033,7.7958,NaN,S
96,988,1,1,"Cavendish, Mrs. Tyrell William (Julia Florence...",female,76.0,1,0,19877,78.8500,C46,S
97,989,0,3,"Makinen, Mr. Kalle Edvard",male,29.0,0,0,STON/O 2. 3101268,7.9250,NaN,S
98,990,1,3,"Braf, Miss. Elin Ester Maria",female,20.0,0,0,347471,7.8542,NaN,S


In [49]:
df.shape

(891, 11)

Na aula anterior, estudamos sobre limpeza e tratamento de dados. Portanto, que tal aplicar seus conhecimentos fazendo a limpeza desta base de dados?

**Agora é com você! Faça a limpeza do Titanic Dataset!**

In [29]:
# Passo 1 - diagnóstico: antes de mexer em qualquer coisa, entender o que temos.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [30]:
# Quantos valores nulos existem por coluna, em quantidade e em porcentagem?
nulos = pd.DataFrame({
    'nulos': df.isna().sum(),
    '%': (df.isna().mean() * 100).round(1),
})
nulos[nulos['nulos'] > 0]

,nulos,%
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


In [31]:
# Linhas repetidas inteiras e IDs repetidos
print('Linhas duplicadas:', df.duplicated().sum())
print('PassengerId repetidos:', df['PassengerId'].duplicated().sum())

Linhas duplicadas: 0
PassengerId repetidos: 0


In [32]:
# Passo 2 - tratamento

# Cabin: 687 nulos (77% da base). Preencher seria inventar dado, então removemos a coluna.
df = df.drop(columns=['Cabin'])

# Age: 177 nulos (20%). É muito para simplesmente descartar as linhas, então imputamos.
# Usamos a MEDIANA por classe e sexo (e não a média geral): a idade típica muda bastante
# entre esses grupos, e a mediana sofre menos com valores extremos.
df['Age'] = df['Age'].fillna(df.groupby(['Pclass', 'Sex'])['Age'].transform('median'))

# Embarked: apenas 2 nulos. Preenchemos com a moda ('S', Southampton).
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Padronização de texto: remove espaços sobrando nas colunas de texto.
for col in ['Name', 'Sex', 'Ticket', 'Embarked']:
    df[col] = df[col].str.strip()

df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [33]:
# Passo 3 - conferência
print('Formato final:', df.shape)
print('Total de nulos:', df.isna().sum().sum())

# Último olhar nos valores estranhos que sobraram. Não vamos removê-los: tarifa zero
# provavelmente é cortesia/tripulação e a tarifa de 512 é real (1a classe), mas é bom
# saber que eles existem antes de olhar qualquer média.
print('Tarifas iguais a zero:', (df['Fare'] == 0).sum())
print('Idade mínima e máxima:', df['Age'].min(), '-', df['Age'].max())
df.head()

Formato final: (891, 11)
Total de nulos: 0
Tarifas iguais a zero: 15
Idade mínima e máxima: 0.42 - 80.0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


## 2. Entendendo tipos de variáveis

### 2.1. Variáveis Qualitativas (Categóricas)

São variáveis que representam grupos ou categorias. Têm valor descritivo, mas não necessariamente matemático.

#### Nominais

Representam rótulos; não possuem ordem.

**Exemplos:** 

- Raça (Branco, Preto, Pardo, etc.);
- Cidade; etc.

### ❓ Pergunta

Há variáveis categórias nominais na nossa base? Se sim, quais?

> **✅ Resposta**
>
> **Sim.** `Sex` (*male*/*female*) e `Embarked` (C, Q, S) são nominais: rótulos sem ordem nenhuma.
> `Name`, `Ticket` e `Cabin` também são texto sem ordem, mas funcionam mais como identificadores
> do que como categorias de análise.
>
> `Survived` também é nominal binária. Está codificada como 0/1, mas 1 não é "maior" que 0, é
> outra categoria.

#### Ordinais

Possuem uma ordem; um senso de progressão, mas não possuem dimensão matemática real.

**Exemplos:** 

- Escolaridade (Ensino Fundamental, Médio, Superior);
- Planos na Netflix (Padrão com anúncios, Padrão sem anúncios, Premium); etc.

### ❓ Pergunta

Há variáveis categóricas ordinais na nossa base? Se sim, quais?

> **✅ Resposta**
>
> **Sim: `Pclass`.** Primeira, segunda e terceira classe têm ordem clara, da melhor para a pior
> acomodação. Mas a diferença entre 1ª e 2ª não é "do mesmo tamanho" que entre 2ª e 3ª: o número
> aqui é um rótulo ordenado, não uma medida.

### ❓ Pergunta

Faz sentido calcular a média de uma variável qualitativa? E fazer uma contagem de possíveis valores?

> **✅ Resposta**
>
> **Média: não.** Não existe "média entre Southampton e Cherbourg". Quando aparece um número numa
> variável qualitativa, ele é apenas um código.
>
> **Contagem e frequência: sim.** É exatamente assim que se descreve uma qualitativa, com
> `value_counts` e moda.
>
> ⚠️ Exceção prática: variáveis binárias codificadas em 0/1, como `Survived`. A média delas é a
> proporção de 1s, ou seja, a taxa de sobrevivência (38,4% na base).

### 2.2. Variáveis Quantitativas

Representam medidas numéricas reais.

#### Discretas

São valores inteiros contáveis.

**Exemplos:**

- Número de filhos;
- Quantidade de itens numa compra.

### ❓ Pergunta

Há variáveis quantitativas discretas na nossa base de dados? Se sim, quais?

> **✅ Resposta**
>
> **Sim:** `SibSp` e `Parch`. São contagens de pessoas, então só assumem inteiros (0, 1, 2, ...).
> Não existe "1,5 irmão".
>
> `PassengerId` também é um número inteiro, mas é identificador, não medida: somar ou tirar média
> dele não significa nada.

#### Contínuas

São valores pertencentes ao conjunto dos reais. Podem assumir qualquer valor dentro de um intervalo.

**Exemplos:**

- Altura;
- Renda;

### ❓ Pergunta

Há variáveis quantitativas discretas na nossa base de dados? Se sim, quais?

> **✅ Resposta**
>
> **Sim:** `Age` e `Fare`.
>
> A idade aceita frações (há bebês com 0,42 ano na base) e a tarifa vai de 0 a 512,33 com
> centavos. As duas podem assumir qualquer valor dentro de um intervalo.

### ❓ Pergunta

Faz sentido calcular a média de uma variável quantitativa? E fazer uma contagem de possíveis valores?

> **✅ Resposta**
>
> **Média: sim.** É a principal medida de posição de uma quantitativa, ao lado de mediana e
> quartis.
>
> **Contagem de valores: depende.** Para discretas com poucos valores possíveis (`SibSp`,
> `Parch`), `value_counts` é informativo. Para contínuas como `Fare`, quase todo valor é único e a
> contagem não diz nada. Nesse caso o caminho é agrupar em faixas (histograma) ou olhar os
> quartis.

## 3. Estatística descritiva

### 3.1. Contagem e frequência

Entender a quantidade de valores em cada categoria de dados é uma etapa relevante para entender sua representatividade e balanceamento dentro da base de dados - evitando que as análises reproduzam um potencial viés existente na base de dados.

Para fazer a contagem de valores, o Pandas possui o comando `value_counts`:

In [34]:
df['Sex'].value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

Valores brutos podem ser difíceis de interpretar, pricipalmente se forem muito grandes. Portanto, pode ser conveniente avaliar a **frequência** com que determinadas categorias aparecem - isto é, a quantidade de ocorrências em relação ao total.

In [35]:
df['Sex'].value_counts(normalize=True) 

Sex
male      0.647587
female    0.352413
Name: proportion, dtype: float64

A **frequência** normalmente terá valores de 0 a 1. Porém podemos transformar isso em percentual multiplicando por 100!

In [36]:
df['Sex'].value_counts(normalize=True)*100

Sex
male      64.758698
female    35.241302
Name: proportion, dtype: float64

### ❓ Pergunta

Há mais homens do que mulheres na nossa base. Como isso pode impactar a nossa análise?

> **✅ Resposta**
>
> A base tem **64,8% homens e 35,2% mulheres**, então qualquer estatística geral é puxada pelo
> grupo maior. A taxa de sobrevivência global (38,4%) fica bem mais perto da dos homens (18,9%) do
> que da das mulheres (74,2%), e esconde justamente a diferença mais importante da base.
>
> Na prática: comparar sempre **taxas dentro de cada grupo**, nunca contagens absolutas entre
> grupos de tamanhos diferentes, e desconfiar de conclusão tirada só do número agregado.

### 3.2. Distribuição: Média, mediana e quartis

São estatísticas quantitativas que ajudam a posicionar dados quantitativos de maneira relativa, de forma a entender sua distribuição.

**Quartis**

Representam os valores limítrofes de faixas de distribuição que contém exatamente 1/4 da quantidade total de elementos (por isso o nome). São utilizados para entender em que faixas de valores os elementos estão concentrados. Por exemplo, na distribuição fictícia abaixo:

- Mínimo = 0.00
- Q1 (25%) = 0.50 -> Valor mais alto entre os 25% valores mais baixos
- Q2 (50%) = 0.63 -> Valor mais alto entre os 50% valores mais baixos
- Q3 (75%) = 0.77 -> Valor mais alto entre os 75% valores mais baixos
- Máximo/Q4 (100%) = 1.00 -> Valor mais alto entre os 100% valores mais baixos

In [37]:
# Quartis
df['Fare'].quantile([0.25, 0.5, 0.75])

0.25     7.9104
0.50    14.4542
0.75    31.0000
Name: Fare, dtype: float64

### ❓ Pergunta

O que podemos afirmar sobre a distribuição dos valores na situação ilustrada?

> **✅ Resposta**
>
> Q1 = 7,91, mediana = 14,45 e Q3 = 31,00, enquanto o máximo é 512,33.
>
> Ou seja: metade dos passageiros pagou menos de 14,5, os 50% do meio estão espremidos numa faixa
> estreita, e a distância de Q3 até o máximo é enorme.
>
> Isso descreve uma distribuição **assimétrica à direita**: concentrada nos valores baixos, com
> uma cauda longa de tarifas altas, candidatas a outliers.

**Mediana:**

É o valor que fica na posição central de uma lista ordenada de valores. Por exemplo, na lista [1, 2, 3, 4, 5], a mediana é o '3'. É também representada pelo quartil Q3 (50%).
Por ser o valor central, nos ajuda a estabelecer se qualquer valor analisado está entre os 50% mais altos ou entre os 50% mais baixos do conjunto universo de elementos, proporcionando um senso relativo de distribuição.

In [38]:
# Mediana
df['Fare'].median()

np.float64(14.4542)

In [39]:
# Também a mediana
df['Fare'].quantile(0.76390)

np.float64(32.20040429999998)

**Média:**

Representa a razão entre a soma dos valores e a quantidade de elementos. É uma maneira de metrificar o quão "pesados" os dados são em relação a si mesmos, comumente de maneira comparativa à mediana.

In [40]:
# Média
df['Fare'].mean()

np.float64(32.204207968574636)

### ❓ Pergunta

O que significa quando a média é maior que a mediana? E quando é mais baixa?

> **✅ Resposta**
>
> * **Média > mediana:** assimetria à direita. Os valores altos puxam a média para cima. É o caso
>   de `Fare`, com média 32,20 contra mediana 14,45.
> * **Média < mediana:** assimetria à esquerda, quando os valores baixos é que puxam.
> * **Média ≈ mediana:** distribuição aproximadamente simétrica.
>
> Como a mediana é robusta a outliers e a média não, a distância entre as duas funciona como um
> termômetro rápido de assimetria.

No Pandas, essas métricas podem ser observadas individualmente, porém o comando `describe` traz uma combinação de todas elas.

In [41]:
df['Fare'].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

### ❓ Pergunta

O que você consegue inferir sobre a variável `Fare` analisando somente os resultados acima?

> **✅ Resposta**
>
> São 891 tarifas, com média 32,20 e mediana 14,45 (a média é mais que o dobro), desvio-padrão
> 49,69, **maior que a própria média**, mínimo 0 e máximo 512,33.
>
> Leitura: distribuição fortemente assimétrica à direita, em que poucos passageiros pagaram muito
> (provavelmente 1ª classe) e a maioria pagou pouco. A média **não** representa o passageiro
> típico; aqui mediana e quartis descrevem melhor.
>
> Os 15 valores iguais a 0 e o teto de 512,33 merecem investigação: podem ser cortesias,
> tripulação ou bilhetes de grupo lançados numa linha só.

## 4. Agregações e métricas agregadas

Analisar variáveis de maneira isolada pode ser útil para entender balanceamento, representatividade, distribuições e até mesmo identificar outliers. Contudo, em uma análise aprofundada, queremos ser capazes de entender também como determinados parâmetros se comportam em relação a outros. Neste sentido, as **agregações** possibilitam que observemos de maneira simultânea um grupo de parâmetros.

### 4.1. Agrupamentos

Úteis principalmente quando precisamos analisar a relação entre variáveis qualitativas e quantitativas.

Imagine que, enquanto investigamos o dataset em questão, a seguinte pergunta apareça: **Como a taxa de sobrevivência se relaciona com os sexo biológico dos passageiros?**

Podemos usar o comando `groupby` para agrupar a média de sobrevivência por sexo e descobrir:

In [42]:
# taxa de sobrevivência por sexo
df.groupby('Sex')['Survived'].mean().reset_index(name='survival_rate')

,Sex,survival_rate
0,female,0.742038
1,male,0.188908


É possível agrupar métricas de variáveis quantitativas por múltiplas variáveis qualitativas, como no exemplo abaixo:

In [43]:
# média de idade por classe e sexo
df.groupby(['Pclass','Sex'])['Age'].mean().unstack()

Sex,female,male
Pclass,,
1,34.648936,41.060820
2,28.703947,30.678981
3,21.677083,26.099193


Também é possível aggregar múltiplas métricas de variáveis quantitativas em relação à variáveis qualitativas. Porém, para isso, é necessário utilizar o comand `agg`.

In [44]:
df.groupby('Pclass').agg(
    passengers=('PassengerId','count'),
    avg_age=('Age','mean'),
    median_fare=('Fare','median')
).reset_index()

,Pclass,passengers,avg_age,median_fare
0,1,216,38.270463,60.2875
1,2,184,29.863207,14.2500
2,3,491,24.802281,8.0500


### 🏋️ Exercício

Descubra a taxa média de sobrevivência por local de embarque e classe comercial.

In [45]:
# Taxa média de sobrevivência por local de embarque (linhas) e classe comercial (colunas)
df.groupby(['Embarked', 'Pclass'])['Survived'].mean().unstack().round(3)

Pclass,1,2,3
Embarked,,,
C,0.694,0.529,0.379
Q,0.500,0.667,0.375
S,0.589,0.463,0.190


> **✅ Resposta**
>
> ```
> Pclass        1      2      3
> Embarked
> C         0.694  0.529  0.379
> Q         0.500  0.667  0.375
> S         0.589  0.463  0.190
> ```
>
> A tabela conta duas histórias ao mesmo tempo:
>
> * **A classe pesa mais que o porto.** Em qualquer porto de embarque, a 1ª classe sobreviveu mais
>   que a 3ª: 69,4% contra 37,9% em Cherbourg, 58,9% contra 19,0% em Southampton.
> * **Cherbourg parece o melhor porto, mas o motivo é a composição.** Dos 168 que embarcaram lá,
>   85 eram de 1ª classe. Em Southampton, 353 dos 646 eram de 3ª. O porto não salvava ninguém: ele
>   só indica quem estava a bordo.
>
> E um alerta de leitura: em Queenstown a 1ª classe aparece com 50% e a 2ª com 66,7%, números que
> parecem ótimos. Só que embarcaram lá **2 pessoas de 1ª classe e 3 de 2ª**. Taxa calculada sobre
> 2 pessoas não é estatística, é anedota. Antes de acreditar em qualquer média, confira o tamanho
> do grupo com um `crosstab` de contagem.

### 4.2. Tabelas cruzadas

Quando queremos verificar a frequência relativa entre categorias específicas de duas ou mais variáveis qualitativas, as **tabelas cruzadas** são a melhor opção.

Por exemplo, se quisermos saber a contagem de pessoas que havia em cada classe comercial por gênero:

In [46]:
pd.crosstab(df['Pclass'], df['Sex'], margins=True)

Sex,female,male,All
Pclass,,,
1,94,122,216
2,76,108,184
3,144,347,491
All,314,577,891


Também é possível observar a frequência relativa de acordo com o índice (nesse caso a classe comercial):

In [47]:
pd.crosstab(df['Pclass'], df['Sex'], margins=True, normalize='index')

Sex,female,male
Pclass,,
1,0.435185,0.564815
2,0.413043,0.586957
3,0.293279,0.706721
All,0.352413,0.647587


### ❓ Pergunta

O que você consegue inferir a partir da análise de frequência de gênero por classe comercial?

> **✅ Resposta**
>
> A composição por gênero muda conforme a classe: na 1ª e na 2ª a divisão homem/mulher é de 56/44
> e 59/41, enquanto a 3ª classe é **70,7% masculina**. E ela sozinha concentra 491 dos 891
> passageiros.
>
> Ou seja, o desbalanceamento de sexo da base vem em boa parte da 3ª classe.
>
> Consequência para a análise: `Sex` e `Pclass` não podem ser lidos isoladamente, porque parte do
> efeito atribuído a um pode ser do outro. Isso se chama **confundimento**, e é por isso que vale
> cruzar as duas variáveis em vez de olhar uma de cada vez.

### 4.3. Correlações

A **correlação** é uma métrica de influência entre variáveis quantitativas. Valores próximos de 1 indicam que as variáveis têm **correlação positiva**, isto é, elas se influenciam de maneira proporcional (crescem juntas e decrescem juntas). Valores próximos de -1 indicam **correlação negativa**, isto é, elas se influenciam de maneira insamente proporcional (enquanto uma cresce, a outra decresce). Já valores próximos de 0 indicam que as variáveis têm pouca influência uma sobre a outra.

In [48]:
df[['Age','Fare','Survived']].corr()

,Age,Fare,Survived
Age,1.000000,0.122692,-0.059579
Fare,0.122692,1.000000,0.257307
Survived,-0.059579,0.257307,1.000000


### ❓ Pergunta

O que é possível dizer a partir da correlação entre as colunas `Age`, `Fare` e `Survived`?

> **✅ Resposta**
>
> * **`Fare` × `Survived` = 0,26.** Positiva e fraca/moderada: quem pagou mais teve mais chance de
>   sobreviver. A tarifa funciona aqui como um *proxy* da classe social.
> * **`Age` × `Survived` = -0,06.** Praticamente nula. Isso **não** significa que idade não
>   importa: a relação não é linear (crianças se salvaram mais, adultos jovens menos) e os efeitos
>   opostos se cancelam.
> * **`Age` × `Fare` = 0,12.** Fraca. Passageiros mais velhos pagaram um pouco mais, em média.
>
> Dois lembretes: a correlação mede apenas relação **linear** entre variáveis **quantitativas**, e
> correlação **não** implica causalidade.